# DFT расчёт HEA L12 (32 атома) на GPU

**Важно:**
1. **Среда выполнения -> Сменить тип -> GPU**.
2. Загрузи `HEA_L12_SQS_32atoms.cif` и папку `pseudo`.

Этот ноутбук устанавливает `nvfortran` (NVIDIA HPC SDK), обходит баги `configure` и собирает QE с GPU.

In [ ]:
# 1. Установка NVIDIA HPC SDK (компилятор nvfortran)
!rm -rf q-e* nvhpc* nvhpc.tar.gz
!apt-get update -qq
!apt-get install -y build-essential gfortran libopenmpi-dev libfftw3-dev libopenblas-dev liblapack-dev libxml2-dev wget > /dev/null
!pip install -q pymatgen

print("Скачивание NVIDIA HPC SDK (это займет время)...")
# Прямая ссылка на архив
!wget -q https://developer.download.nvidia.com/hpc-sdk/24.9/nvhpc_2024_249_Linux_x86_64_cuda_12.6.tar.gz -O nvhpc.tar.gz
!tar xf nvhpc.tar.gz

print("Установка (тихий режим)...")
!cd nvhpc_2024_249_Linux_x86_64_cuda_12.6 && ./install -silent

import os
nvfortran_path = "/opt/nvidia/hpc_sdk/Linux_x86_64/24.9/compilers/bin/nvfortran"
if os.path.exists(nvfortran_path):
    print("✅ NVHPC установлен.")
    # Настраиваем окружение
    os.environ['PATH'] = '/opt/nvidia/hpc_sdk/Linux_x86_64/24.9/compilers/bin:' + os.environ.get('PATH', '')
    os.environ['LD_LIBRARY_PATH'] = '/opt/nvidia/hpc_sdk/Linux_x86_64/24.9/compilers/lib:/opt/nvidia/hpc_sdk/Linux_x86_64/24.9/cuda/12.6/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
    os.environ['CUDA_HOME'] = '/opt/nvidia/hpc_sdk/Linux_x86_64/24.9/cuda/12.6'
else:
    print("❌ Ошибка установки NVHPC.")

In [ ]:
# 2. Скачивание и компиляция QE
import os

print("Скачивание QE 7.3...")
!wget -q https://github.com/QEF/q-e/archive/refs/tags/qe-7.3.tar.gz -O qe-7.3.tar.gz
!tar xf qe-7.3.tar.gz

print("Патч configure (отключаем проверку версии)...")
conf_path = "q-e-qe-7.3/configure"
with open(conf_path, 'r') as f:
    content = f.read()
content = content.replace('as_fn_error', 'echo "[PATCHED]" # as_fn_error')
with open(conf_path, 'w') as f:
    f.write(content)

print("Компиляция с GPU (nvfortran + OpenBLAS + FFTW3)...")
os.chdir("q-e-qe-7.3")

# Явно указываем OpenBLAS и FFTW3, чтобы избежать конфликта с MKL
os.environ['BLAS_LIBS'] = '-lopenblas'
os.environ['LAPACK_LIBS'] = '-llapack -lblas'

nvf = "/opt/nvidia/hpc_sdk/Linux_x86_64/24.9/compilers/bin/nvfortran"
!./configure --enable-cuda --with-cuda=$CUDA_HOME --with-blas=openblas --with-lapack=external --with-fft=fftw3 F90={nvf} CC=gcc

if os.path.exists("make.inc"):
    print("\nСборка pw.x...")
    !make pw -j 4
    
    if os.path.exists("PW/src/pw.x"):
        print("\n✅ Компиляция успешна!")
    else:
        print("\n❌ ОШИБКА сборки.")
else:
    print("\n❌ ОШИБКА конфигурации.")

os.chdir("..")

In [ ]:
# 3. Генерация input и запуск
from pymatgen.core import Structure
import re, os

cif_file = "HEA_L12_SQS_32atoms.cif"
pseudo_map = {
    "Al": "Al.pbesol-n-kjpaw_psl.1.0.0.UPF",
    "Ti": "Ti.upf",
    "Cr": "Cr.upf",
    "Fe": "Fe.pbesol-spn-kjpaw_psl.0.2.1.UPF",
    "Co": "Co.upf",
    "Ni": "Ni.upf"
}
masses = {"Al": 26.98, "Ti": 47.87, "Cr": 52.00, "Fe": 55.85, "Co": 58.93, "Ni": 58.69}

struct = Structure.from_file(cif_file)

inp = f"""&CONTROL
    calculation = 'vc-relax'
    prefix = 'HEA_L12_32'
    pseudo_dir = './pseudo'
    outdir = './tmp'
    etot_conv_thr = 1.0d-4
    forc_conv_thr = 1.0d-3
    nstep = 200
    verbosity = 'high'
    device = 'GPU'
/
&SYSTEM
    ibrav = 0
    nat = {struct.num_sites}
    ntyp = {len(pseudo_map)}
    ecutwfc = 60
    ecutrho = 480
    occupations = 'smearing'
    smearing = 'methfessel-paxton'
    degauss = 0.02
    input_dft = 'pbesol'
/
&ELECTRONS
    conv_thr = 1.0d-8
    mixing_beta = 0.3
    electron_maxstep = 200
/
&IONS
    ion_dynamics = 'bfgs'
/
&CELL
    cell_dynamics = 'bfgs'
    press = 0.0
/
ATOMIC_SPECIES
"""

for el, fname in pseudo_map.items():
    inp += f" {el}  {masses[el]}  {fname}\n"

inp += "CELL_PARAMETERS angstrom\n"
for v in struct.lattice.matrix:
    inp += f" {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n"

inp += "ATOMIC_POSITIONS crystal\n"
for site in struct:
    sym = site.species.elements[0].symbol
    fc = site.frac_coords
    inp += f" {sym}  {fc[0]:.6f} {fc[1]:.6f} {fc[2]:.6f}\n"

inp += "K_POINTS automatic\n 4 4 4  0 0 0\n"

with open("vc-relax.in", "w") as f:
    f.write(inp)

print(f"Создан vc-relax.in ({struct.num_sites} атомов).")

pw_path = "./q-e-qe-7.3/PW/src/pw.x"
if not os.path.exists(pw_path):
    print("ОШИБКА: pw.x не найден.")
else:
    print(f"Запуск на GPU...")
    !{pw_path} < vc-relax.in 2>&1 | tee vc-relax.out

    with open("vc-relax.out", "r") as f:
        out = f.read()
    energy = re.search(r'!\s+total energy\s+=\s+([\-\d\.]+)\s+Ry', out)
    if energy:
        print(f"\nЭнергия: {energy.group(1)} Ry")